# Comprehensive Evaluation - trained model + ensemble vs. heuristics

Stress-tests the trained Packing Transformer plus an **ensemble agent** that combines
the model with every heuristic and replays whichever member won the voyage. The
ensemble is **mathematically guaranteed to never lose** to any individual member -
it picks the best result per voyage by construction.

Sections:
1. Setup -- load model, prepare datasets, sanity check.
2. Macro benchmark -- 20 Alexandria-realistic voyages, all algorithms.
3. Voyage-size sweep -- 50 / 80 / 120 / 200 items.
4. Container-type sweep -- 20GP, 40GP, 40HC, 45HC.
5. BR academic benchmark.
6. Edge cases.
7. Constraint validation.
8. Centre-of-Gravity analysis.
9. Inference performance.
10. Visual gallery -- 3D side-by-side {Bottom-Left, Ours, Ensemble}.
11. Thesis summary -- CSV + one-page figure.

All artefacts land in `benchmarks/out/` and a zip at `/kaggle/working/`.


## 1. Setup

In [ ]:
import os, sys, subprocess
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')

REPO_URL = 'https://github.com/Seif-Sameh/loading-service.git'
BRANCH = 'main'

# 1) Clone repo if not already on disk
if os.path.isdir('loading-service'):
    subprocess.run(['rm', '-rf', 'loading-service'], check=True)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, 'loading-service'], check=True)
os.chdir('loading-service')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# 2) Install deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--upgrade-strategy', 'only-if-needed',
    '-r', 'requirements.txt', '-r', 'requirements-rl.txt',
], check=True)

# 3) Datasets
if not os.path.isdir('/tmp/wadaboa-bpp'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Wadaboa/3d-bpp.git', '/tmp/wadaboa-bpp'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_datasets',
                '--wadaboa-pkl', '/tmp/wadaboa-bpp/data/products.pkl'], check=True)
print('environment ready')

In [ ]:
# 4) Locate the trained model.
# Priority: explicit path > /kaggle/input/<dataset>/* > local models/ dir.
import shutil, glob
import torch

MODEL_PATH_OVERRIDE = None  # set to your uploaded path if you want to be explicit

def _find_model() -> str:
    if MODEL_PATH_OVERRIDE and os.path.isfile(MODEL_PATH_OVERRIDE):
        return MODEL_PATH_OVERRIDE
    # Kaggle input dataset (uploaded model)
    candidates = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
    if candidates:
        # Make sure models/ exists before copying
        os.makedirs('models', exist_ok=True)
        target = f'models/{os.path.basename(candidates[0])}'
        if not os.path.isfile(target):
            shutil.copy(candidates[0], target)
        return target
    # Local repo models/ dir
    local = sorted(glob.glob('models/*.pt'))
    if local:
        return local[0]
    raise FileNotFoundError(
        'No .pt checkpoint found. Upload your trained model via Kaggle Add Data → Upload, '
        'or set MODEL_PATH_OVERRIDE to its absolute path.'
    )

MODEL_PATH = _find_model()
print(f'using model: {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1024:.1f} KB)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

In [ ]:
# 5) Common imports + global plot style
import time, statistics, json, math, random
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from app.catalog.loader import get_container, list_containers, list_cargo_presets
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig
from app.data.br_loader import (
    br_container_to_isolike, br_problem_to_items, list_br_problems,
)
from app.algorithms import get_algorithm
from app.algorithms.base import solve
from app.algorithms.rl.ppo_agent import PPOPackingAgent
from app.algorithms.ensemble import EnsembleAgent
from app.constraints.imdg import imdg_violations
from app.constraints.cog import CoGTracker
from app.schemas import HazmatClass

plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
PALETTE = {
    'bl':              '#1a4d7a',
    'extreme_points':  '#2a6aa6',
    'baf':             '#5a8bc9',
    'bssf':            '#94b8d9',
    'blsf':            '#c5d8e6',
    'ours':            '#ef5350',
    'ensemble':        '#2e7d32',
}
ALG_LABELS = {
    'bl': 'Bottom-Left', 'extreme_points': 'Extreme Points', 'baf': 'Best Area Fit',
    'bssf': 'Best Shortest Side', 'blsf': 'Best Longest Side',
    'ours': 'Ours (PPO+Transformer)',
    'ensemble': 'Ensemble (best-of-runs)',
}
ALL_ALGS = ['bl', 'extreme_points', 'baf', 'bssf', 'blsf', 'ours', 'ensemble']
OUT = Path('benchmarks/out'); OUT.mkdir(parents=True, exist_ok=True)

OURS = PPOPackingAgent(weights_path=MODEL_PATH, device=device)

def get_alg(code: str):
    if code == 'ours': return OURS
    # Ensemble holds per-voyage state, so build a fresh one for each call.
    if code == 'ensemble': return EnsembleAgent(ppo_agent=OURS)
    return get_algorithm(code)

print('ready -- algorithms:', list(ALG_LABELS.keys()))


In [ ]:
# 6) Sanity check — one quick voyage to confirm the model loads and runs
cont = get_container('40HC')
items = AlexandriaSampler(SamplerConfig(n_items=50, strategy='mixed', seed=1)).sample()
print(f'sanity voyage: 40HC × 50 items')
for code in ALL_ALGS:
    res, _ = solve(algorithm=get_alg(code), container=cont, items=items)
    print(f'  {ALG_LABELS[code]:<24}  util {res.kpis.utilization*100:>6.2f}%  placed {len(res.placements):>3}/{len(items)}  {res.elapsed_ms:>5.0f} ms')

## 2. Macro benchmark — 30 Alexandria-realistic voyages

This is the headline number for the thesis. 30 fresh voyages, 100 items each, 40HC
container. Every algorithm runs on every voyage; we record full KPIs.

Bump `N_VOYAGES` to 50-100 for the final thesis run; 30 is fine for inspection.

In [ ]:
N_VOYAGES = 20            # bump to 50+ for thesis-grade
ITEMS_PER_VOYAGE = 80     # ensemble runs each member; smaller is faster

_macro_alex = AlexandriaSampler(SamplerConfig(n_items=ITEMS_PER_VOYAGE, strategy='mixed', seed=None))
_macro_voyages = [(get_container('40HC'), _macro_alex.sample()) for _ in range(N_VOYAGES)]

def run_suite(voyages, algs):
    rows = []
    for vi, (c, its) in enumerate(voyages):
        for code in algs:
            res, _ = solve(algorithm=get_alg(code), container=c, items=its)
            k = res.kpis
            rows.append({
                'voyage': vi,
                'algorithm': code,
                'container': c.code.value,
                'n_items': len(its),
                'placed': len(res.placements),
                'placed_pct': 100*len(res.placements)/max(len(its),1),
                'util_pct': 100*k.utilization,
                'weight_pct': 100*k.weight_used,
                'cog_long_dev': k.cog_long_dev,
                'cog_lat_dev': k.cog_lat_dev,
                'cog_vert_frac': k.cog_vert_frac,
                'unstable_count': k.unstable_count,
                'imdg_violation_count': k.imdg_violation_count,
                'lifo_violation_count': k.lifo_violation_count,
                'stack_violation_count': k.stack_violation_count,
                'overloaded_count': k.overloaded_count,
                'elapsed_ms': res.elapsed_ms,
            })
    return pd.DataFrame(rows)

macro_df = run_suite(_macro_voyages, ALL_ALGS)
macro_df.to_csv(OUT/'macro.csv', index=False)
print(f'wrote {OUT/"macro.csv"}: {len(macro_df)} rows')

In [ ]:
# Aggregate & display
agg = macro_df.groupby('algorithm').agg(
    util_mean=('util_pct', 'mean'),
    util_std=('util_pct', 'std'),
    util_min=('util_pct', 'min'),
    util_max=('util_pct', 'max'),
    placed_mean=('placed_pct', 'mean'),
    weight_mean=('weight_pct', 'mean'),
    cog_long_abs=('cog_long_dev', lambda s: s.abs().mean()),
    cog_lat_abs=('cog_lat_dev', lambda s: s.abs().mean()),
    cog_vert=('cog_vert_frac', 'mean'),
    unstable=('unstable_count', 'mean'),
    imdg=('imdg_violation_count', 'mean'),
    ms=('elapsed_ms', 'mean'),
).reindex(ALL_ALGS)
agg.index = [ALG_LABELS[c] for c in agg.index]
print(agg.round(2).to_string())

In [ ]:
# Boxplots: util, placed_pct, CoG, time
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, col, title, ylab in [
    (axes[0], 'util_pct',     'Utilisation %',         'util %'),
    (axes[1], 'placed_pct',   'Items placed %',        'placed %'),
    (axes[2], 'cog_long_dev', 'Longitudinal CoG dev.', 'dev (signed)'),
    (axes[3], 'elapsed_ms',   'Wall time per voyage',  'ms'),
]:
    data = [macro_df[macro_df.algorithm == a][col].values for a in ALL_ALGS]
    bp = ax.boxplot(data, patch_artist=True, labels=[ALG_LABELS[a].replace(' ','\n') for a in ALL_ALGS])
    for patch, code in zip(bp['boxes'], ALL_ALGS):
        patch.set_facecolor(PALETTE[code]); patch.set_alpha(0.85)
    ax.set_title(title); ax.set_ylabel(ylab)
    ax.tick_params(axis='x', labelsize=8)
fig.suptitle(f'Macro benchmark — {N_VOYAGES} voyages × {ITEMS_PER_VOYAGE} items, 40HC', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(OUT/'macro_boxplots.png', dpi=130, bbox_inches='tight'); plt.show()

In [ ]:
# Bar plot of mean util with error bars
fig, ax = plt.subplots(figsize=(9, 4.5))
means = [macro_df[macro_df.algorithm == a]['util_pct'].mean() for a in ALL_ALGS]
stds  = [macro_df[macro_df.algorithm == a]['util_pct'].std()  for a in ALL_ALGS]
x = np.arange(len(ALL_ALGS))
ax.bar(x, means, yerr=stds, color=[PALETTE[a] for a in ALL_ALGS], alpha=0.85, capsize=6)
ax.set_xticks(x); ax.set_xticklabels([ALG_LABELS[a] for a in ALL_ALGS], rotation=15)
ax.set_ylabel('mean utilisation %  (±1σ)')
ax.set_title('Utilisation by algorithm')
ax.axhline(max(means), color='black', linestyle='--', alpha=0.4, linewidth=1)
for xi, m in zip(x, means):
    ax.text(xi, m + 0.5, f'{m:.2f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.savefig(OUT/'macro_bar.png', dpi=130, bbox_inches='tight'); plt.show()

## 3. Voyage-size sweep

How does utilisation scale with voyage size? Smaller voyages have *less* potential
utilisation (less cargo to fill the container); larger voyages saturate.

In [ ]:
SIZE_VOYAGES_PER_SIZE = 6
VOYAGE_SIZES = [50, 80, 120, 200]

size_rows = []
for n in VOYAGE_SIZES:
    smp = AlexandriaSampler(SamplerConfig(n_items=n, strategy='mixed', seed=None))
    voyages = [(get_container('40HC'), smp.sample()) for _ in range(SIZE_VOYAGES_PER_SIZE)]
    df = run_suite(voyages, ALL_ALGS)
    df['size'] = n
    size_rows.append(df)
size_df = pd.concat(size_rows, ignore_index=True)
size_df.to_csv(OUT/'voyage_size_sweep.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for code in ALL_ALGS:
    sub = size_df[size_df.algorithm == code].groupby('size')['util_pct'].mean()
    axes[0].plot(sub.index, sub.values, 'o-', label=ALG_LABELS[code], color=PALETTE[code], linewidth=2)
    sub2 = size_df[size_df.algorithm == code].groupby('size')['placed_pct'].mean()
    axes[1].plot(sub2.index, sub2.values, 's-', label=ALG_LABELS[code], color=PALETTE[code], linewidth=2)
axes[0].set_xlabel('items per voyage'); axes[0].set_ylabel('util %'); axes[0].set_title('Utilisation vs voyage size'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('items per voyage'); axes[1].set_ylabel('placed %'); axes[1].set_title('Placement rate vs voyage size'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'voyage_size_sweep.png', dpi=130, bbox_inches='tight'); plt.show()

## 4. Container-type sweep

Generalisation across container sizes: same algorithm, different containers.

In [ ]:
CONTAINER_CODES = ['20GP', '40GP', '40HC', '45HC']
VOYAGES_PER_CONT = 5

cont_rows = []
for code in CONTAINER_CODES:
    cont = get_container(code)
    smp = AlexandriaSampler(SamplerConfig(n_items=80, strategy='mixed', seed=None))
    voyages = [(cont, smp.sample()) for _ in range(VOYAGES_PER_CONT)]
    df = run_suite(voyages, ALL_ALGS)
    cont_rows.append(df)
container_df = pd.concat(cont_rows, ignore_index=True)
container_df.to_csv(OUT/'container_sweep.csv', index=False)

fig, ax = plt.subplots(figsize=(11, 4.5))
ind = np.arange(len(CONTAINER_CODES))
width = 1.0 / (len(ALL_ALGS) + 1)
for i, code in enumerate(ALL_ALGS):
    means = [container_df[(container_df.algorithm == code) & (container_df.container == cc)]['util_pct'].mean() for cc in CONTAINER_CODES]
    ax.bar(ind + i*width, means, width, label=ALG_LABELS[code], color=PALETTE[code], alpha=0.85)
ax.set_xticks(ind + width*(len(ALL_ALGS)-1)/2); ax.set_xticklabels(CONTAINER_CODES)
ax.set_xlabel('container type'); ax.set_ylabel('mean util %'); ax.set_title('Utilisation by container type (120-item voyages)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.savefig(OUT/'container_sweep.png', dpi=130, bbox_inches='tight'); plt.show()

## 5. BR academic benchmark (Bischoff & Ratcliff)

Standard 3D-BPP testbed — 100 boxes, container ≈ 20GP. Numbers here are directly
comparable to published papers (DeepPack3D heuristics: 50-58 %, GOPT: 76 %).

In [ ]:
BR_PROBLEMS = list_br_problems()
N_BR_PROBLEMS = 15  # bump to 50+ for thesis-grade

br_voyages = []
for p in BR_PROBLEMS[:N_BR_PROBLEMS]:
    cc = br_container_to_isolike(p)
    its = br_problem_to_items(p)
    br_voyages.append((cc, its))
br_df = run_suite(br_voyages, ALL_ALGS)
br_df.to_csv(OUT/'br_benchmark.csv', index=False)

br_agg = br_df.groupby('algorithm').agg(
    util_mean=('util_pct', 'mean'), util_std=('util_pct', 'std'),
    placed_mean=('placed_pct', 'mean'), ms=('elapsed_ms', 'mean'),
).reindex(ALL_ALGS)
br_agg.index = [ALG_LABELS[c] for c in br_agg.index]
print(f'BR1 — first {N_BR_PROBLEMS} problems')
print(br_agg.round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 4.5))
means = br_agg['util_mean'].values
stds = br_agg['util_std'].values
x = np.arange(len(ALL_ALGS))
ax.bar(x, means, yerr=stds, color=[PALETTE[a] for a in ALL_ALGS], alpha=0.85, capsize=6)
ax.set_xticks(x); ax.set_xticklabels([ALG_LABELS[a] for a in ALL_ALGS], rotation=15)
ax.set_ylabel('mean util %'); ax.set_title(f'BR1 academic benchmark — {N_BR_PROBLEMS} problems')
ax.axhline(76, color='red', linestyle='--', alpha=0.5, linewidth=1, label='GOPT (paper) ≈ 76 %')
ax.legend()
for xi, m in zip(x, means): ax.text(xi, m + 1, f'{m:.1f}', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout(); plt.savefig(OUT/'br_benchmark.png', dpi=130, bbox_inches='tight'); plt.show()

## 6. Edge cases

Six adversarial / specialty scenarios that probe specific behaviours of the constraint
layer + algorithms:
1. Pure Wadaboa (all real, no presets — small parcels only)
2. Pure presets (pallets / drums / IBCs)
3. Hazmat-heavy (50 % class-8 corrosives) — IMDG segregation pressure
4. All heavy machinery (1.5 t each) — payload + floor-load pressure
5. Reefer-required (all items requires_reefer=True) — compatibility check
6. Single huge item + many small fillers

In [ ]:
from app.catalog.loader import get_cargo_preset
from app.schemas import CargoItem, Dimensions, FragilityClass, HazmatClass

def edge_pure_wadaboa(seed):
    s = AlexandriaSampler(SamplerConfig(n_items=150, strategy='real', seed=seed))
    return get_container('40HC'), s.sample()

def edge_pure_presets(seed):
    s = AlexandriaSampler(SamplerConfig(n_items=80, strategy='presets', seed=seed))
    return get_container('40HC'), s.sample()

def edge_hazmat_heavy(seed):
    rng = random.Random(seed)
    items = []
    for i in range(100):
        if rng.random() < 0.5:
            it = get_cargo_preset('hazmat_corrosive_drum', item_id=f'haz-{i:03d}')
        else:
            it = get_cargo_preset(rng.choice(['eur_pallet_light', 'carton_large']), item_id=f'reg-{i:03d}')
        items.append(it)
    return get_container('40HC'), items

def edge_all_machinery(seed):
    items = [get_cargo_preset('machinery_small', item_id=f'm-{i:03d}') for i in range(20)]
    return get_container('40HC'), items

def edge_all_reefer(seed):
    items = [get_cargo_preset('reefer_fruit_pallet', item_id=f'rf-{i:03d}') for i in range(40)]
    return get_container('40RF'), items

def edge_single_giant(seed):
    big = CargoItem(
        id='giant', preset_code=None, label='giant',
        dimensions=Dimensions(length_mm=4000, width_mm=2000, height_mm=2000),
        weight_kg=5000.0, fragility=FragilityClass.NORMAL,
    )
    fillers = AlexandriaSampler(SamplerConfig(n_items=80, strategy='mixed', seed=seed)).sample()
    return get_container('40HC'), [big] + fillers

EDGE_TESTS = [
    ('pure_wadaboa',  'Pure Wadaboa (small parcels only)',     edge_pure_wadaboa),
    ('pure_presets',  'Pure presets (pallets / drums / IBCs)',  edge_pure_presets),
    ('hazmat_heavy',  '50% hazmat (IMDG segregation)',          edge_hazmat_heavy),
    ('all_machinery', 'All heavy machinery (1.5 t each)',       edge_all_machinery),
    ('all_reefer',    'All reefer items (40RF container)',      edge_all_reefer),
    ('single_giant',  'Single giant + small fillers',           edge_single_giant),
]

EDGE_VOYAGES_PER_TEST = 3
edge_rows = []
for code, label, fn in EDGE_TESTS:
    voyages = [fn(seed=s) for s in range(EDGE_VOYAGES_PER_TEST)]
    df = run_suite(voyages, ALL_ALGS)
    df['edge_test'] = code
    df['edge_label'] = label
    edge_rows.append(df)
edge_df = pd.concat(edge_rows, ignore_index=True)
edge_df.to_csv(OUT/'edge_cases.csv', index=False)

edge_agg = edge_df.groupby(['edge_label', 'algorithm']).agg(
    util_mean=('util_pct','mean'), placed_mean=('placed_pct','mean'),
    imdg=('imdg_violation_count','mean'), unstable=('unstable_count','mean'),
    ms=('elapsed_ms','mean'),
).round(2)
print(edge_agg.to_string())


In [ ]:
# Heatmap: util across (edge case, algorithm)
pivot = edge_df.groupby(['edge_label', 'algorithm'])['util_pct'].mean().unstack().reindex(columns=ALL_ALGS)
pivot.columns = [ALG_LABELS[c] for c in pivot.columns]
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=80)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=15)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.1f}', ha='center', va='center', fontsize=9, color='black')
ax.set_title('Mean utilisation % by edge-case and algorithm')
plt.colorbar(im, ax=ax, label='util %')
plt.tight_layout(); plt.savefig(OUT/'edge_heatmap.png', dpi=130, bbox_inches='tight'); plt.show()

## 7. Constraint-validation summary

How often does each algorithm produce IMDG / stability / LIFO / stack violations? The hard
feasibility mask should keep IMDG and reefer violations at zero; only soft-constraint
metrics (CoG drift, stack order, stability) can vary.

In [ ]:
all_runs_df = pd.concat([macro_df, size_df, container_df, edge_df], ignore_index=True)
viol = all_runs_df.groupby('algorithm').agg(
    voyages=('voyage', 'count'),
    imdg_total=('imdg_violation_count', 'sum'),
    imdg_voyages=('imdg_violation_count', lambda s: (s>0).sum()),
    unstable_total=('unstable_count', 'sum'),
    unstable_voyages=('unstable_count', lambda s: (s>0).sum()),
    overloaded_total=('overloaded_count', 'sum'),
    lifo_total=('lifo_violation_count', 'sum'),
    stack_total=('stack_violation_count', 'sum'),
).reindex(ALL_ALGS)
viol.index = [ALG_LABELS[c] for c in viol.index]
print('Total violation counts across ALL benchmarks (Macro + sizes + containers + edge):')
print(viol.to_string())
viol.to_csv(OUT/'constraint_violations.csv')

## 8. Centre-of-Gravity analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col, title, ideal in [
    (axes[0], 'cog_long_dev', 'Longitudinal CoG deviation (target = 0)',  0.0),
    (axes[1], 'cog_lat_dev',  'Lateral CoG deviation (target = 0)',       0.0),
    (axes[2], 'cog_vert_frac','Vertical CoG fraction (target ≤ 0.40)',    0.40),
]:
    data = [macro_df[macro_df.algorithm == a][col].values for a in ALL_ALGS]
    bp = ax.boxplot(data, patch_artist=True, labels=[ALG_LABELS[a].replace(' ','\n') for a in ALL_ALGS])
    for patch, code in zip(bp['boxes'], ALL_ALGS):
        patch.set_facecolor(PALETTE[code]); patch.set_alpha(0.85)
    ax.axhline(ideal, color='black', linestyle='--', alpha=0.5)
    ax.set_title(title); ax.tick_params(axis='x', labelsize=8)
fig.suptitle('Centre-of-Gravity distribution (macro benchmark)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(OUT/'cog_analysis.png', dpi=130, bbox_inches='tight'); plt.show()

In [ ]:
# CoG evolution during packing on one example voyage — shows how the CoG drifts as items go in.
demo_cont, demo_items = _macro_voyages[0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for code in ['bl', 'extreme_points', 'ours']:
    res, _ = solve(algorithm=get_alg(code), container=demo_cont, items=demo_items)
    cog = CoGTracker(container=demo_cont)
    long_devs, util_running = [], []
    items_by_id = {it.id: it for it in demo_items}
    cum_vol = 0
    for p in res.placements:
        cog.add(p, items_by_id[p.item_id].weight_kg)
        long_devs.append(cog.longitudinal_deviation)
        cum_vol += p.rotated_dimensions.volume_mm3
        util_running.append(100*cum_vol/demo_cont.internal.volume_mm3)
    axes[0].plot(long_devs, label=ALG_LABELS[code], color=PALETTE[code], linewidth=2)
    axes[1].plot(util_running, label=ALG_LABELS[code], color=PALETTE[code], linewidth=2)
axes[0].axhline(0.10, color='gray', linestyle=':', alpha=0.5, label='warning band')
axes[0].axhline(-0.10, color='gray', linestyle=':', alpha=0.5)
axes[0].set_xlabel('placement #'); axes[0].set_ylabel('long. CoG dev'); axes[0].set_title('CoG drift during packing'); axes[0].legend()
axes[1].set_xlabel('placement #'); axes[1].set_ylabel('util %'); axes[1].set_title('Utilisation growth'); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT/'cog_evolution.png', dpi=130, bbox_inches='tight'); plt.show()

## 9. Inference performance

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for code in ALL_ALGS:
    sub = macro_df[macro_df.algorithm == code]
    ax.scatter(sub.placed.values, sub.elapsed_ms.values, label=ALG_LABELS[code],
               color=PALETTE[code], s=42, alpha=0.7, edgecolor='black', linewidth=0.4)
ax.set_xlabel('items placed'); ax.set_ylabel('wall time (ms)')
ax.set_title('Inference speed: items placed vs total wall-time')
ax.legend(fontsize=8); ax.set_yscale('log')
plt.tight_layout(); plt.savefig(OUT/'inference_timing.png', dpi=130, bbox_inches='tight'); plt.show()

speed = macro_df.groupby('algorithm').apply(lambda s: (s.placed.sum() / (s.elapsed_ms.sum() / 1000))).reindex(ALL_ALGS)
speed.index = [ALG_LABELS[c] for c in speed.index]
print(f'\nThroughput (placements / second):')
print(speed.round(1).to_string())

## 10. Visual gallery — best / median / worst voyages

In [ ]:
# Pick the best/median/worst voyages (by ours util%)
ours_macro = macro_df[macro_df.algorithm == 'ours'].sort_values('util_pct').reset_index()
best_voyage_idx = int(ours_macro.iloc[-1]['voyage'])
med_voyage_idx = int(ours_macro.iloc[len(ours_macro)//2]['voyage'])
worst_voyage_idx = int(ours_macro.iloc[0]['voyage'])

def draw_box(ax, x, y, z, dx, dy, dz, color):
    verts = [
        [(x,y,z),(x+dx,y,z),(x+dx,y+dy,z),(x,y+dy,z)],
        [(x,y,z+dz),(x+dx,y,z+dz),(x+dx,y+dy,z+dz),(x,y+dy,z+dz)],
        [(x,y,z),(x+dx,y,z),(x+dx,y,z+dz),(x,y,z+dz)],
        [(x,y+dy,z),(x+dx,y+dy,z),(x+dx,y+dy,z+dz),(x,y+dy,z+dz)],
        [(x,y,z),(x,y+dy,z),(x,y+dy,z+dz),(x,y,z+dz)],
        [(x+dx,y,z),(x+dx,y+dy,z),(x+dx,y+dy,z+dz),(x+dx,y,z+dz)],
    ]
    ax.add_collection3d(Poly3DCollection(verts, alpha=0.55, facecolor=color, edgecolor='black', linewidth=0.2))

def render_voyage(ax, container, items, algo_code, title):
    res, _ = solve(algorithm=get_alg(algo_code), container=container, items=items)
    cs = plt.cm.viridis(np.linspace(0, 1, max(len(res.placements), 1)))
    for i, p in enumerate(res.placements):
        draw_box(ax, p.position.x_mm, p.position.z_mm, p.position.y_mm,
                 p.rotated_dimensions.length_mm, p.rotated_dimensions.width_mm, p.rotated_dimensions.height_mm,
                 cs[i])
    ax.set_xlim(0, container.internal.length_mm)
    ax.set_ylim(0, container.internal.width_mm)
    ax.set_zlim(0, container.internal.height_mm)
    ax.set_box_aspect([container.internal.length_mm, container.internal.width_mm, container.internal.height_mm])
    ax.set_title(f'{title}\n{ALG_LABELS[algo_code]}: util {res.kpis.utilization*100:.1f}%, placed {len(res.placements)}/{len(items)}', fontsize=9)

fig = plt.figure(figsize=(16, 10))
for row, (vidx, label) in enumerate([(best_voyage_idx, 'BEST voyage for ours'),
                                       (med_voyage_idx, 'MEDIAN voyage'),
                                       (worst_voyage_idx, 'WORST voyage for ours')]):
    cont, items = _macro_voyages[vidx]
    for col, code in enumerate(['bl', 'ours', 'ensemble']):
        ax = fig.add_subplot(3, 3, row*3 + col + 1, projection='3d')
        render_voyage(ax, cont, items, code, label if col == 0 else '')
fig.suptitle('Best / Median / Worst voyages — three algorithms side-by-side', fontweight='bold', y=1.0)
plt.tight_layout(); plt.savefig(OUT/'gallery_3d.png', dpi=130, bbox_inches='tight'); plt.show()

## 11. Thesis summary — one-page everything

In [ ]:
# Aggregate every benchmark into a single table
summary = pd.DataFrame({
    'macro_util_mean':     macro_df.groupby('algorithm')['util_pct'].mean(),
    'macro_util_std':      macro_df.groupby('algorithm')['util_pct'].std(),
    'macro_placed_mean':   macro_df.groupby('algorithm')['placed_pct'].mean(),
    'br_util_mean':        br_df.groupby('algorithm')['util_pct'].mean(),
    'edge_util_mean':      edge_df.groupby('algorithm')['util_pct'].mean(),
    'cog_long_abs_mean':   macro_df.groupby('algorithm')['cog_long_dev'].apply(lambda s: s.abs().mean()),
    'cog_vert_mean':       macro_df.groupby('algorithm')['cog_vert_frac'].mean(),
    'imdg_total':          all_runs_df.groupby('algorithm')['imdg_violation_count'].sum(),
    'unstable_total':      all_runs_df.groupby('algorithm')['unstable_count'].sum(),
    'mean_ms_per_voyage':  macro_df.groupby('algorithm')['elapsed_ms'].mean(),
}).reindex(ALL_ALGS)
summary.index = [ALG_LABELS[c] for c in summary.index]
summary = summary.round(2)
summary.to_csv(OUT/'thesis_summary.csv')
print('THESIS SUMMARY  (csv at', OUT/'thesis_summary.csv', ')')
print(summary.to_string())

In [ ]:
# Final 4-panel figure — drop straight into the thesis chapter
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel 1: macro utilization bar with error bars
x = np.arange(len(ALL_ALGS))
means = [macro_df[macro_df.algorithm == a]['util_pct'].mean() for a in ALL_ALGS]
stds  = [macro_df[macro_df.algorithm == a]['util_pct'].std()  for a in ALL_ALGS]
axes[0,0].bar(x, means, yerr=stds, color=[PALETTE[a] for a in ALL_ALGS], alpha=0.85, capsize=5)
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels([ALG_LABELS[a] for a in ALL_ALGS], rotation=20, fontsize=8)
axes[0,0].set_ylabel('util % (mean ± σ)'); axes[0,0].set_title(f'(a) Macro benchmark — Alexandria-realistic, {N_VOYAGES} voyages × {ITEMS_PER_VOYAGE} items')
for xi, m in zip(x, means): axes[0,0].text(xi, m + 0.5, f'{m:.1f}', ha='center', fontsize=8, fontweight='bold')

# Panel 2: BR benchmark
br_means = br_df.groupby('algorithm')['util_pct'].mean().reindex(ALL_ALGS).values
br_stds = br_df.groupby('algorithm')['util_pct'].std().reindex(ALL_ALGS).values
axes[0,1].bar(x, br_means, yerr=br_stds, color=[PALETTE[a] for a in ALL_ALGS], alpha=0.85, capsize=5)
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels([ALG_LABELS[a] for a in ALL_ALGS], rotation=20, fontsize=8)
axes[0,1].set_ylabel('util %'); axes[0,1].set_title(f'(b) BR1 academic benchmark — {N_BR_PROBLEMS} problems')
axes[0,1].axhline(76, color='red', linestyle='--', alpha=0.5, linewidth=1)
axes[0,1].text(len(ALL_ALGS)-1, 77, 'GOPT (paper) ≈ 76 %', ha='right', fontsize=8, color='red')
for xi, m in zip(x, br_means): axes[0,1].text(xi, m + 1, f'{m:.1f}', ha='center', fontsize=8, fontweight='bold')

# Panel 3: voyage size scaling
for code in ALL_ALGS:
    sub = size_df[size_df.algorithm == code].groupby('size')['util_pct'].mean()
    axes[1,0].plot(sub.index, sub.values, 'o-', label=ALG_LABELS[code], color=PALETTE[code], linewidth=2)
axes[1,0].set_xlabel('items per voyage'); axes[1,0].set_ylabel('mean util %')
axes[1,0].set_title('(c) Scaling with voyage size'); axes[1,0].legend(fontsize=7, loc='lower right')

# Panel 4: edge-case heatmap mini
pivot_small = edge_df.groupby(['edge_label', 'algorithm'])['util_pct'].mean().unstack().reindex(columns=ALL_ALGS)
pivot_small.columns = [ALG_LABELS[c].split()[0] for c in pivot_small.columns]
im = axes[1,1].imshow(pivot_small.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=70)
axes[1,1].set_xticks(range(len(pivot_small.columns))); axes[1,1].set_xticklabels(pivot_small.columns, rotation=15, fontsize=8)
axes[1,1].set_yticks(range(len(pivot_small.index))); axes[1,1].set_yticklabels(pivot_small.index, fontsize=8)
for i in range(len(pivot_small.index)):
    for j in range(len(pivot_small.columns)):
        axes[1,1].text(j, i, f'{pivot_small.values[i,j]:.0f}', ha='center', va='center', fontsize=8)
axes[1,1].set_title('(d) Edge cases — util % heatmap')
plt.colorbar(im, ax=axes[1,1], shrink=0.7)

fig.suptitle('Trained model vs. heuristics — one-page summary', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(OUT/'thesis_one_page.png', dpi=140, bbox_inches='tight'); plt.show()
print('thesis_one_page.png saved to', OUT)

In [ ]:
# Bundle every artifact into a single zip for easy download
import shutil
zip_path = '/kaggle/working/evaluation_results.zip' if os.path.isdir('/kaggle/working') else 'evaluation_results.zip'
shutil.make_archive(zip_path[:-4], 'zip', OUT)
print(f'all CSVs and PNGs zipped at {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)')
print('Files:')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name:<32}  {f.stat().st_size/1024:>6.1f} KB')